Excercise 1: Duplicate Detection and Removal

In [18]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder
from sklearn.impute import SimpleImputer
import os
print ("All libraries loaded! Ready for preprocessing.")

All libraries loaded! Ready for preprocessing.


In [3]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Define the path to your folder in Google Drive where you want to store the data
# You might need to create this folder manually in your Drive if it doesn't exist.
# For example, create a folder named 'colab_data' in your Drive root.
drive_path = '/content/drive/MyDrive/colab_notebooks/developers_institute/'

Mounted at /content/drive


In [4]:
import os

full_file_path = os.path.join(drive_path,'/content/drive/MyDrive/colab_notebooks/developers_institute/titanic_train.csv')
print(f"Attempting to load data from: {full_file_path}")
titanic_raw = pd.read_csv(full_file_path)
titanic_raw.columns = titanic_raw.columns.str.lower()

print (f"Shape: {titanic_raw.shape}")
titanic_raw.info()
print('')
titanic_raw.describe()

Attempting to load data from: /content/drive/MyDrive/colab_notebooks/developers_institute/titanic_train.csv
Shape: (891, 12)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   passengerid  891 non-null    int64  
 1   survived     891 non-null    int64  
 2   pclass       891 non-null    int64  
 3   name         891 non-null    object 
 4   sex          891 non-null    object 
 5   age          714 non-null    float64
 6   sibsp        891 non-null    int64  
 7   parch        891 non-null    int64  
 8   ticket       891 non-null    object 
 9   fare         891 non-null    float64
 10  cabin        204 non-null    object 
 11  embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB



,passengerid,survived,pclass,age,sibsp,parch,fare
count,891.000000,891.000000,891.000000,714.000000,891.000000,891.000000,891.000000
mean,446.000000,0.383838,2.308642,29.699118,0.523008,0.381594,32.204208
std,257.353842,0.486592,0.836071,14.526497,1.102743,0.806057,49.693429
min,1.000000,0.000000,1.000000,0.420000,0.000000,0.000000,0.000000
25%,223.500000,0.000000,2.000000,20.125000,0.000000,0.000000,7.910400
50%,446.000000,0.000000,3.000000,28.000000,0.000000,0.000000,14.454200
75%,668.500000,1.000000,3.000000,38.000000,1.000000,0.000000,31.000000
max,891.000000,1.000000,3.000000,80.000000,8.000000,6.000000,512.329200


In [5]:
df = titanic_raw.copy()
df.columns = df.columns.str.lower() # Ensure df columns are lowercase
print ('Checking for Duplicates')
print(f"Total rows: {len(df)}")
print(f"Exact Duplicate rows: {df.duplicated().sum()}")

Checking for Duplicates
Total rows: 891
Exact Duplicate rows: 0


In [6]:
ID_COLS = ['passengerid', 'name', 'ticket', 'cabin'] # Use lowercase column names
model_view = df.drop (columns=ID_COLS)
model_view.head(10)

,survived,pclass,sex,age,sibsp,parch,fare,embarked
0,0,3,male,22.0,1,0,7.2500,S
1,1,1,female,38.0,1,0,71.2833,C
2,1,3,female,26.0,0,0,7.9250,S
3,1,1,female,35.0,1,0,53.1000,S
4,0,3,male,35.0,0,0,8.0500,S
5,0,3,male,NaN,0,0,8.4583,Q
6,0,1,male,54.0,0,0,51.8625,S
7,0,3,male,2.0,3,1,21.0750,S
8,1,3,female,27.0,0,2,11.1333,S
9,1,2,female,14.0,1,0,30.0708,C


In [7]:
df.columns = df.columns.str.lower() # Ensure df has lowercase columns
model_view.columns = model_view.columns.str.lower() # Ensure model_view has lowercase columns

print(f"Columns a model would actually see:\n  {list(model_view.columns)}")
print(f"\nDuplicate rows now: {model_view.duplicated().sum()}")
print(f"Rows involved in a collision: {model_view.duplicated(keep=False).sum()}")

# Who are these "duplicates"? Put the names back and look.
collisions = df[model_view.duplicated(keep=False)].sort_values(['pclass', 'age', 'fare'])
print("\nThe first three colliding pairs, with names restored:")
print(collisions[['name', 'pclass', 'sex', 'age', 'fare', 'survived']].head(6).to_string())
print('')
print("Decision not to drop duplicates, different people")

Columns a model would actually see:
  ['survived', 'pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked']

Duplicate rows now: 111
Rows involved in a collision: 167

The first three colliding pairs, with names restored:
                              name  pclass     sex   age   fare  survived
369  Aubart, Mme. Leontine Pauline       1  female  24.0  69.30         1
641           Sagesser, Mlle. Emma       1  female  24.0  69.30         1
252      Stead, Mr. William Thomas       1    male  62.0  26.55         0
555             Wright, Mr. George       1    male  62.0  26.55         0
633  Parr, Mr. William Henry Marsh       1    male   NaN   0.00         0
815               Fry, Mr. Richard       1    male   NaN   0.00         0

Decision not to drop duplicates, different people


Excercise 2: Handling Missing Values

In [8]:
# === STEP 2: ANALYZE MISSING VALUES ===
print("STEP 2: Missing Value Analysis")
print("=" * 40)

# Count missing values
missing_counts = df.isnull().sum()
missing_pct = (missing_counts / len(df)) * 100

# Show only columns with missing values
missing_df = pd.DataFrame({
    'Missing Count': missing_counts,
    'Missing %': missing_pct
})
missing_df = missing_df[missing_df['Missing Count'] > 0].sort_values('Missing %', ascending=False)

print("\nColumns with missing values:")
for col, row in missing_df.iterrows():
    bar = '█' * int(row['Missing %'] / 3)
    print(f"  {col:15} {row['Missing Count']:4.0f} ({row['Missing %']:5.1f}%) {bar}")



STEP 2: Missing Value Analysis

Columns with missing values:
  cabin            687 ( 77.1%) █████████████████████████
  age              177 ( 19.9%) ██████
  embarked           2 (  0.2%) 


Exericise 3 Feature Engineering

In [9]:
df['family_size'] = df['sibsp'] + df['parch'] + 1
print("created 'family_size' = sibsp + parch + 1")
print("\nFamily size distribution:")
print(df['family_size'].value_counts().sort_index())

created 'family_size' = sibsp + parch + 1

Family size distribution:
family_size
1     537
2     161
3     102
4      29
5      15
6      22
7      12
8       6
11      7
Name: count, dtype: int64


In [10]:
df['title'] = df['name'].str.extract(r',\s*([^.]+)\.')

print("Extracted titles:")
print(df['title'].value_counts())

Extracted titles:
title
Mr              517
Miss            182
Mrs             125
Master           40
Dr                7
Rev               6
Col               2
Mlle              2
Major             2
Ms                1
Mme               1
Don               1
Lady              1
Sir               1
Capt              1
the Countess      1
Jonkheer          1
Name: count, dtype: int64


In [11]:
# Simplify rare titles into groups
# Too many categories = harder for models to learn

title_mapping = {
    'Mr': 'Mr',
    'Miss': 'Miss',
    'Mrs': 'Mrs',
    'Master': 'Master',
    # Map rare/foreign titles
    'Dr': 'Rare',
    'Rev': 'Rare',
    'Col': 'Rare',
    'Major': 'Rare',
    'Mlle': 'Miss',      # Mademoiselle = Miss
    'Mme': 'Mrs',        # Madame = Mrs
    'Ms': 'Mrs',
    'Capt': 'Rare',
    'Don': 'Rare',
    'Jonkheer': 'Rare',
    'Lady': 'Rare',
    'Sir': 'Rare',
    'the Countess': 'Rare',
}

df['title'] = df['title'].map(title_mapping)
df['title'] = df['title'].fillna('Rare')  # Any unmapped titles

print("Simplified titles (5 categories):")
print(df['title'].value_counts())

Simplified titles (5 categories):
title
Mr        517
Miss      184
Mrs       127
Master     40
Rare       23
Name: count, dtype: int64


Excercise 4: Outlier Detection and Handling

In [12]:
# Calculate IQR for Fare
Q1 = df['fare'].quantile(0.25)
Q3 = df['fare'].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

print("IQR Analysis for Fare:")
print(f"  Q1 (25th percentile): ${Q1:.2f}")
print(f"  Q3 (75th percentile): ${Q3:.2f}")
print(f"  IQR: ${IQR:.2f}")
print(f"\n  Lower bound: ${lower_bound:.2f}")
print(f"  Upper bound: ${upper_bound:.2f}")

# Count outliers
outliers = df[(df['fare'] < lower_bound) | (df['fare'] > upper_bound)]
print(f"\n  Outliers detected: {len(outliers)} ({len(outliers)/len(df)*100:.1f}% of data)")

IQR Analysis for Fare:
  Q1 (25th percentile): $7.91
  Q3 (75th percentile): $31.00
  IQR: $23.09

  Lower bound: $-26.72
  Upper bound: $65.63

  Outliers detected: 116 (13.0% of data)


In [13]:
# Cap outliers at 98th percentile (our choice for Titanic)
# Why 98th? Less aggressive than IQR, keeps most extreme values reasonable

fare_cap = df['fare'].quantile(0.98)
outliers_count = (df['fare'] > fare_cap).sum()

print(f"Capping fare at 98th percentile: ${fare_cap:.2f}")
print(f"Values to be capped: {outliers_count}")
print(f"\nBefore capping:")
print(f"  Max fare: ${df['fare'].max():.2f}")

# Apply the cap
df['fare'] = df['fare'].clip(upper=fare_cap)

print(f"\nAfter capping:")
print(f"  Max fare: ${df['fare'].max():.2f}")

Capping fare at 98th percentile: $211.34
Values to be capped: 17

Before capping:
  Max fare: $512.33

After capping:
  Max fare: $211.34


Excercise 5: Data Standardization and Normalization

In [14]:
# Apply StandardScaler to age and fare
scaler = StandardScaler()

df['age_scaled'] = scaler.fit_transform(df[['age']])
df['fare_scaled'] = scaler.fit_transform(df[['fare']])

print("After StandardScaler:")
print(f"  Age:  mean={df['age_scaled'].mean():.4f}, std={df['age_scaled'].std():.4f}")
print(f"  Fare: mean={df['fare_scaled'].mean():.4f}, std={df['fare_scaled'].std():.4f}")
print("\nBoth now centered at 0 with std of 1!")

After StandardScaler:
  Age:  mean=0.0000, std=1.0007
  Fare: mean=-0.0000, std=1.0006

Both now centered at 0 with std of 1!


In [15]:
# Compare original vs scaled
print("Comparison (first 5 rows):")
print(df[['age', 'age_scaled', 'fare', 'fare_scaled']].head())

Comparison (first 5 rows):
    age  age_scaled     fare  fare_scaled
0  22.0   -0.530377   7.2500    -0.587437
1  38.0    0.571831  71.2833     1.018110
2  26.0   -0.254825   7.9250    -0.570512
3  35.0    0.365167  53.1000     0.562189
4  35.0    0.365167   8.0500    -0.567378


Excercise 6: Feature Encoding

In [17]:
# STEP 5: Feature Encoding
print("STEP 5: Feature Encoding")
print("=" * 40)

# First, see what categorical columns we have
print("\nCategorical columns to encode:")
print(f"  sex: {df['sex'].unique()}")
print(f"  embarked: {df['embarked'].unique()}")
print(f"  title: {df['title'].unique()}")

STEP 5: Feature Encoding

Categorical columns to encode:
  sex: ['male' 'female']
  embarked: ['S' 'C' 'Q' nan]
  title: ['Mr' 'Mrs' 'Miss' 'Master' 'Rare']


In [22]:
# Label Encoding for binary 'sex' column
# Only 2 values, no ordering issue

df['sex_encoded'] = df['sex'].map({'male': 0, 'female': 1})

print("Label encoded 'sex':")
print(df[['sex', 'sex_encoded']].head())

Label encoded 'sex':
      sex  sex_encoded
0    male            0
1  female            1
2  female            1
3  female            1
4    male            0


In [23]:
import pandas as pd

# One-Hot Encoding for 'embarked' (3 unordered categories)
# Using drop_first=True to avoid redundancy

embarked_dummies = pd.get_dummies(df['embarked'], prefix='embarked', drop_first=True)

print("One-hot encoded 'embarked':")
print("(drop_first=True: if embarked_Q=0 and embarked_S=0, we know it's C)")
print(embarked_dummies.head())

One-hot encoded 'embarked':
(drop_first=True: if embarked_Q=0 and embarked_S=0, we know it's C)
   embarked_Q  embarked_S
0       False        True
1       False       False
2       False        True
3       False        True
4       False        True


In [24]:
# Add one-hot columns to our dataframe
df = pd.concat([df, embarked_dummies], axis=1)

print(f"Added {len(embarked_dummies.columns)} new columns for 'embarked'")

Added 2 new columns for 'embarked'


In [25]:
# One-Hot Encoding for 'title' (5 categories)
title_dummies = pd.get_dummies(df['title'], prefix='title', drop_first=True)
df = pd.concat([df, title_dummies], axis=1)

print(f"Added {len(title_dummies.columns)} new columns for 'title':")
print(list(title_dummies.columns))

Added 4 new columns for 'title':
['title_Miss', 'title_Mr', 'title_Mrs', 'title_Rare']


Excercise 7: Data Transformation for Age Feature


In [ ]:
# Create age groups using pd.cut()
# This bins continuous age into meaningful categories

bins = [0, 12, 18, 35, 60, 100]
labels = ['Child', 'Teen', 'Young Adult', 'Middle Aged', 'Senior']

df['age_group'] = pd.cut(df['age'], bins=bins, labels=labels)

print("Created 'age_group' from continuous age:")
print(df['age_group'].value_counts())

Created 'age_group' from continuous age:
age_group
Young Adult    535
Middle Aged    195
Teen            70
Child           69
Senior          22
Name: count, dtype: int64
